# Chilli Leaf Disease Classification — DINOv3 + LoRA

This notebook provides a complete, end-to-end research workflow:

1. **Imports**
2. **Configuration** — load YAML config and set seeds for reproducibility
3. **Dataset** — load and visualise the Chilli Leaf Disease dataset
4. **DataLoaders** — create train / val / test loaders with the configured splits
5. **Model** — instantiate DINOv3 ViT-S/16 backbone
6. **LoRA** — apply PEFT LoRA adapters on attention layers
7. **Training Setup** — optimizer, scheduler, loss, early stopping
8. **Training Loop** — full AMP training loop
9. **Validation** — evaluate on the validation split
10. **Metrics** — compute accuracy, precision, recall, F1
11. **Confusion Matrix** — plot and save the confusion matrix
12. **Grad-CAM** — generate Grad-CAM visualisations
13. **Inference** — single-image prediction

Run `pip install -r requirements.txt` first. Ensure CUDA is available before launching heavy cells.

## 1. Imports

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt

from utils.helpers import (
    get_device, get_logger, set_seed, resolve_path_str,
    ensure_dir, count_parameters,
)
from datasets.dataset import (
    ChilliLeafDataset, build_train_transforms, build_eval_transforms,
    create_dataloaders,
)
from models.dinov3_lora import (
    build_model, save_checkpoint, load_checkpoint,
)
from models.model_utils import format_param_count
from utils.metrics import (
    compute_classification_metrics, classification_report_text,
    plot_confusion_matrix, save_metrics_bundle,
)
from utils.gradcam import ViTGradCAM, save_heatmap_visual, load_image_rgb
print(f'Project root: {PROJECT_ROOT}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 2. Configuration

All hyperparameters live in `configs/config.yaml`. Edit that file to change the run.

In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
with open(CONFIG_PATH, "r", encoding="utf-8") as fh:
    config = yaml.safe_load(fh)

set_seed(config["seed"], deterministic=config.get("deterministic", True),
         benchmark=config.get("benchmark", False))

DATASET_DIR = Path(resolve_path_str(PROJECT_ROOT, config["paths"]["dataset_dir"]))
CHECKPOINT_DIR = ensure_dir(resolve_path_str(PROJECT_ROOT, config["paths"]["checkpoint_dir"]))
OUTPUT_DIR = ensure_dir(resolve_path_str(PROJECT_ROOT, config["paths"]["output_dir"]))
LOG_DIR = ensure_dir(resolve_path_str(PROJECT_ROOT, config["paths"]["log_dir"]))

print('Classes :', config['classes'])
print('Dataset :', DATASET_DIR)
print('Outputs :', OUTPUT_DIR)
print('Seeds   :', config['seed'])

## 3. Dataset — load and visualise

In [ ]:
dataset = ChilliLeafDataset(root=DATASET_DIR, class_names=config['classes'])
print(f'Total images: {len(dataset)}')
print(f'Samples per class:')
for cls, n in dataset.class_counts().items():
    print(f'  {cls:<22} {n:>5}')

# Visualise 3 random samples per class
fig, axes = plt.subplots(3, len(config['classes']), figsize=(2.6 * len(config['classes']), 7.5))
rng = np.random.default_rng(config['seed'])
for col, cls in enumerate(config['classes']):
    indices = dataset.class_indices()[cls]
    pick = rng.choice(indices, size=3, replace=False) if len(indices) >= 3 else indices
    for row, idx in enumerate(pick):
        path, label = dataset.samples[idx]
        axes[row, col].imshow(load_image_rgb(path, size=224))
        axes[row, col].set_title(cls, fontsize=8)
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()

## 4. DataLoaders

In [ ]:
train_loader, val_loader, test_loader, class_names = create_dataloaders(
    root=DATASET_DIR,
    image_size=int(config['dataset']['image_size']),
    batch_size=int(config['training']['batch_size']),
    num_workers=int(config['training']['num_workers']),
    pin_memory=bool(config['training'].get('pin_memory', True)),
    train_ratio=float(config['dataset']['train_ratio']),
    val_ratio=float(config['dataset']['val_ratio']),
    test_ratio=float(config['dataset']['test_ratio']),
    seed=int(config['seed']),
    class_names=config['classes'],
)
print(f'Train batches: {len(train_loader)}  ({len(train_loader.dataset)} samples)')
print(f'Val   batches: {len(val_loader)}  ({len(val_loader.dataset)} samples)')
print(f'Test  batches: {len(test_loader)}  ({len(test_loader.dataset)} samples)')
x, y = next(iter(train_loader))
print(f'Batch shape  : {tuple(x.shape)}  dtype={x.dtype}')
print(f'Label range  : {y.min().item()} .. {y.max().item()}')

## 5. Model — DINOv3 ViT-S/16 backbone

In [ ]:
model_cfg = config['model']
model = build_model(
    backbone_name=model_cfg['backbone_name'],
    num_classes=int(config['num_classes']),
    cache_dir=model_cfg.get('cache_dir'),
    lora_config=None,
    freeze_backbone=False,
)
device = get_device(prefer_cuda=True)
model.to(device)
n_total, n_train = count_parameters(model)
print(f'Backbone only  -> total={format_param_count(n_total)}, trainable={format_param_count(n_train)}')

## 6. LoRA — PEFT adapters on attention layers

In [ ]:
model = build_model(
    backbone_name=model_cfg['backbone_name'],
    num_classes=int(config['num_classes']),
    cache_dir=model_cfg.get('cache_dir'),
    lora_config=config.get('lora'),
    freeze_backbone=bool(model_cfg.get('freeze_backbone', True)),
)
model.to(device)
n_total, n_train = count_parameters(model)
print(f'+ LoRA        -> total={format_param_count(n_total)}, trainable={format_param_count(n_train)}')
print(f'Trainable ratio: {100 * n_train / n_total:.4f}%')

## 7. Training Setup — optimizer, scheduler, loss

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from utils.losses import create_loss
from utils.early_stopping import EarlyStopping

decay, no_decay = [], []
for n, p in model.named_parameters():
    if not p.requires_grad:
        continue
    (no_decay if p.ndim == 1 or 'bias' in n else decay).append(p)

optimizer = AdamW(
    [
        {'params': decay,    'weight_decay': float(config['training']['weight_decay'])},
        {'params': no_decay, 'weight_decay': 0.0},
    ],
    lr=float(config['training']['learning_rate']),
    betas=tuple(config['training'].get('betas', [0.9, 0.999])),
    eps=float(config['training'].get('eps', 1e-8)),
)
scheduler = CosineAnnealingLR(
    optimizer,
    T_max=int(config['training']['epochs']),
    eta_min=float(config['training'].get('eta_min', 1e-6)),
)
criterion = create_loss(
    class_weights=None,
    label_smoothing=float(config['training'].get('label_smoothing', 0.0)),
)
early_stopping = EarlyStopping(
    patience=int(config['training']['patience']),
    min_delta=float(config['training']['min_delta']),
    mode='max',
)
scaler = torch.cuda.amp.GradScaler(enabled=bool(config['training'].get('amp', True)))
print('Optimizer  : AdamW (decay vs no-decay param groups)')
print('Scheduler  : CosineAnnealingLR')
print('Loss       : CrossEntropyLoss')
print('EarlyStop  :', early_stopping)

## 8. Training Loop

In [ ]:
from utils.metrics import compute_accuracy

history = []
best_val_acc = -1.0
for epoch in range(1, int(config['training']['epochs']) + 1):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):
            logits = model(x)['logits']
            loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=float(config['training']['grad_clip_max_norm']),
        )
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item() * x.size(0)
        train_correct += (logits.argmax(-1) == y).sum().item()
        train_total += x.size(0)
    scheduler.step()
    train_loss /= max(train_total, 1)
    train_acc = train_correct / max(train_total, 1)

    # Validation
    model.eval()
    val_correct, val_total, val_loss = 0, 0, 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=scaler.is_enabled()):
                logits = model(x)['logits']
                loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
            val_correct += (logits.argmax(-1) == y).sum().item()
            val_total += x.size(0)
    val_loss /= max(val_total, 1)
    val_acc = val_correct / max(val_total, 1)

    history.append({
        'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
        'val_loss': val_loss, 'val_acc': val_acc,
        'lr': optimizer.param_groups[0]['lr'],
    })
    print(f'epoch {epoch:>2}/{config["training"]["epochs"]}  '
          f'train_loss={train_loss:.4f} train_acc={train_acc:.4f}  '
          f'val_loss={val_loss:.4f} val_acc={val_acc:.4f}')

    # Save best
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        save_checkpoint(
            model, optimizer, scheduler, epoch,
            {'val_acc': val_acc}, CHECKPOINT_DIR / 'best_model.pt',
        )
    early_stopping.step(val_acc)
    if early_stopping.should_stop:
        print(f'Early stopping at epoch {epoch}')
        break

save_checkpoint(model, optimizer, scheduler, epoch,
                {'val_acc': val_acc}, CHECKPOINT_DIR / 'last_model.pt')

## 9. Validation

In [ ]:
best = load_checkpoint(CHECKPOINT_DIR / 'best_model.pt', model, map_location=device)
model.eval()
val_correct, val_total = 0, 0
all_logits, all_labels = [], []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device, non_blocking=True)
        logits = model(x)['logits']
        all_logits.append(logits.cpu()); all_labels.append(y)
        val_correct += (logits.argmax(-1).cpu() == y).sum().item()
        val_total += x.size(0)
print(f'Best checkpoint val_acc: {val_correct / max(val_total, 1):.4f}  '
      f'(epoch {best.get("epoch")})')
val_probs = torch.softmax(torch.cat(all_logits), dim=-1).numpy()
val_pred = val_probs.argmax(-1)
val_true = torch.cat(all_labels).numpy()

## 10. Metrics

In [ ]:
metrics = compute_classification_metrics(
    y_true=val_true, y_pred=val_pred, class_names=class_names,
    y_prob=val_probs, average='macro',
)
print(classification_report_text(metrics))
save_metrics_bundle(metrics, OUTPUT_DIR / 'val_metrics.json')
print('Saved:', OUTPUT_DIR / 'val_metrics.json')

## 11. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plot_confusion_matrix(
    y_true=val_true, y_pred=val_pred, class_names=class_names,
    normalize='true', ax=ax,
    title='Validation Confusion Matrix (Normalised)',
)
plt.tight_layout()
cm_path = OUTPUT_DIR / 'val_confusion_matrix.png'
plt.savefig(cm_path, dpi=150)
plt.show()
print('Saved:', cm_path)

## 12. Grad-CAM

In [ ]:
cam = ViTGradCAM(model)
transform = build_eval_transforms(int(config['dataset']['image_size']))
gradcam_dir = ensure_dir(resolve_path_str(PROJECT_ROOT, config['paths']['gradcam_dir']))

# Visualise 1 sample per class
rng = np.random.default_rng(config['seed'])
shown = 0
for cls_idx, cls_name in enumerate(class_names):
    indices = np.where(val_true == cls_idx)[0]
    if len(indices) == 0:
        continue
    idx = int(rng.choice(indices))
    sample_path, _ = dataset.samples[idx]
    img = load_image_rgb(sample_path, size=int(config['dataset']['image_size']))
    img_t = transform(image=img)['image'] if False else torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
    img_t = torch.nn.functional.normalize(
        img_t, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],
    ).unsqueeze(0).to(device)
    heatmap = cam.generate(img_t, target_class=cls_idx)
    out = gradcam_dir / f'{cls_name}_gradcam.png'
    save_heatmap_visual(img, heatmap, out, title=cls_name)
    print('Saved:', out)
    shown += 1
cam.remove_hooks()
print(f'Generated {shown} Grad-CAM visualisations.')

## 13. Inference — single image prediction

In [ ]:
import json
sample_path, true_label = dataset.samples[0]
img = load_image_rgb(sample_path, size=int(config['dataset']['image_size']))
img_t = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
img_t = torch.nn.functional.normalize(
    img_t, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225],
).unsqueeze(0).to(device)

model.eval()
with torch.no_grad():
    logits = model(img_t)['logits']
    probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()
pred_idx = int(probs.argmax())
result = {
    'image': str(sample_path),
    'true_label': class_names[true_label],
    'predicted_class': class_names[pred_idx],
    'confidence_pct': float(probs[pred_idx] * 100),
    'probabilities_pct': {class_names[i]: float(probs[i] * 100) for i in range(len(class_names))},
}
print(json.dumps(result, indent=2))
with open(OUTPUT_DIR / 'inference_sample.json', 'w', encoding='utf-8') as fh:
    json.dump(result, fh, indent=2)